# Electricity CLP Token Playground

This notebook is a compact playground for CLP-style token chains on the real GiftEval electricity_H_long dataset.

What it does:

1. Sets up Colab/local paths.
2. Downloads and prepares real electricity time-series examples.
3. Defines the tokens used in the experiment directly in the notebook.
4. Shows the chaining logic as simply as possible.
5. Lets you edit one chain and inspect every step.
6. Runs exhaustive search over small token chains and displays a full results table.


## 0. Setup

Run this first. On Colab it clones/pulls kernels_playground and installs the dependencies needed to load GiftEval and run the lightweight models.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import itertools

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()

def run(cmd):
    print("$", " ".join(map(str, cmd)))
    subprocess.check_call(list(map(str, cmd)))

if IN_COLAB:
    os.chdir("/content")
    KERNELS_REPO = Path("/content/kernels_playground")
    if KERNELS_REPO.exists():
        run(["git", "-C", KERNELS_REPO, "pull", "--ff-only"])
    else:
        run(["git", "clone", "https://github.com/chahineNejm/kernels_playground.git", KERNELS_REPO])
    req = KERNELS_REPO / "first_tests" / "requirements.txt"
    run([sys.executable, "-m", "pip", "install", "-q", "-r", req, "pandas", "scikit-learn", "tqdm", "datasets"])
else:
    HERE = Path.cwd()
    candidates = [
        HERE / "kernels_playground",
        HERE.parent / "kernels_playground",
        HERE.parent.parent / "kernels_playground",
        Path(r"C:/Users/ADMIN/Desktop/stage CMS/kernels_playground"),
    ]
    KERNELS_REPO = next((p for p in candidates if p.exists()), candidates[0])

FIRST_TESTS = KERNELS_REPO / "first_tests"
for p in [FIRST_TESTS, KERNELS_REPO]:
    p = str(p.resolve())
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from utils.config import DATASETS
from utils.data import build_examples
from utils.augmentation import uniform_length

np.set_printoptions(precision=4, suppress=True)
print("IN_COLAB:", IN_COLAB)
print("kernels first_tests:", FIRST_TESTS, FIRST_TESTS.exists())


## 1. Download Electricity Data

This loads electricity_H_long from GiftEval Parquet through your existing kernels_playground helpers. Keep the first run small; kernel and ridge tokens train one model per chain.


In [ ]:
CONFIG_NAME = "electricity_H_long"
START = 0
STOP = 120
STEP = 4
HISTORY_LEN = 2000
FUTURE_LEN = 720
TRAIN_FRAC = 0.70
SEED = 0

raw = build_examples(
    config=CONFIG_NAME,
    start=START,
    stop=STOP,
    step=STEP,
    dataset_name=DATASETS["eval"],
)

fixed = uniform_length(
    raw,
    target_len=HISTORY_LEN,
    min_len=HISTORY_LEN // 2,
    keys=("history",),
    seed=SEED,
    verbose=True,
)
fixed = [r for r in fixed if len(r["future"]) >= FUTURE_LEN]
if len(fixed) < 6:
    raise ValueError("Need at least 6 usable examples. Increase STOP or reduce STEP.")

H_ALL = np.stack([np.asarray(r["history"], dtype=np.float32) for r in fixed])
F_ALL = np.stack([np.asarray(r["future"][:FUTURE_LEN], dtype=np.float32) for r in fixed])

n = H_ALL.shape[0]
n_train = max(3, int(TRAIN_FRAC * n))
H_TRAIN, F_TRAIN = H_ALL[:n_train], F_ALL[:n_train]
H_TEST, F_TEST = H_ALL[n_train:], F_ALL[n_train:]

print("all:", H_ALL.shape, F_ALL.shape)
print("train:", H_TRAIN.shape, F_TRAIN.shape)
print("test:", H_TEST.shape, F_TEST.shape)
assert H_TEST.shape[0] > 0, "No test rows. Increase STOP or reduce TRAIN_FRAC."


In [ ]:
def plot_examples(H, F, n_show=4):
    n_show = min(n_show, H.shape[0])
    fig, axes = plt.subplots(n_show, 1, figsize=(12, 2.4 * n_show), squeeze=False)
    for i, ax in enumerate(axes[:, 0]):
        h_tail = H[i, -min(300, H.shape[1]):]
        ax.plot(np.arange(-len(h_tail), 0), h_tail, color="0.55", label="history tail")
        ax.plot(np.arange(F.shape[1]), F[i], color="black", label="future")
        ax.axvline(0, color="tab:blue", ls=":", lw=1)
        ax.set_title(f"sample {i}")
        ax.grid(alpha=0.2)
        if i == 0:
            ax.legend()
    fig.tight_layout()

plot_examples(H_TRAIN, F_TRAIN)


## 2. Minimal Chaining State

This state keeps separate train and test arrays. Tokens transform both sides using only history-derived statistics. Model tokens fit on train residuals and predict test residuals.

Chaining rule:

current target = target base - cumulative predictions

The next model fits that current target.


In [ ]:
def mase(actual, forecast, history):
    mae = np.mean(np.abs(actual - forecast), axis=1)
    scale = np.mean(np.abs(np.diff(history, axis=1)), axis=1)
    scale = np.where(scale < 1e-8, 1.0, scale)
    return float(np.mean(mae / scale))


class PlaygroundState:
    def __init__(self, H_train, F_train, H_test, F_test):
        self.H_train = H_train.astype(np.float32)
        self.F_train = F_train.astype(np.float32)
        self.H_test = H_test.astype(np.float32)
        self.F_test = F_test.astype(np.float32)
        self.features = {
            "raw_train": self.H_train.copy(),
            "raw_test": self.H_test.copy(),
        }
        self.target_base_train = self.F_train.copy()
        self.target_base_test = self.F_test.copy()
        self.current_target_train = self.F_train.copy()
        self.current_target_test = self.F_test.copy()
        self.pred_train = []
        self.pred_test = []
        self.pred_names = []
        self.steps = []
        self.final_forecast = None
        self.mase = None

    def cumulative_train(self):
        return sum(self.pred_train) if self.pred_train else np.zeros_like(self.F_train)

    def cumulative_test(self):
        return sum(self.pred_test) if self.pred_test else np.zeros_like(self.F_test)

    def push_prediction(self, pred_train, pred_test, name):
        self.pred_train.append(pred_train.astype(np.float32))
        self.pred_test.append(pred_test.astype(np.float32))
        self.pred_names.append(name)
        self.current_target_train = self.target_base_train - self.cumulative_train()
        self.current_target_test = self.target_base_test - self.cumulative_test()
        self.features["last_prediction_train"] = pred_train
        self.features["last_prediction_test"] = pred_test
        self.features["current_residual_train"] = self.current_target_train
        self.features["current_residual_test"] = self.current_target_test

    def log(self, token_name):
        self.steps.append({
            "token": token_name,
            "features": {k: tuple(v.shape) for k, v in self.features.items() if hasattr(v, "shape")},
            "n_predictions": len(self.pred_test),
            "train_residual_norm": float(np.linalg.norm(self.current_target_train)),
            "test_residual_norm": float(np.linalg.norm(self.current_target_test)),
            "mase": self.mase,
        })


## 3. Tokens Used In This Notebook

The tokens are intentionally small. Cleaning tokens read the latest cleaned layer when available, so they are chainable. Feature tokens write model_input_train/test. Model tokens write predictions and update residuals.


In [ ]:
def latest_clean_pair(state):
    if "clean_train" in state.features:
        return state.features["clean_train"], state.features["clean_test"]
    return state.features["raw_train"], state.features["raw_test"]


class NormalizeToken:
    name = "normalize"
    kind = "cleaning"
    def apply(self, state):
        Xtr, Xte = latest_clean_pair(state)
        mu_tr = Xtr.mean(axis=1, keepdims=True); sig_tr = Xtr.std(axis=1, keepdims=True)
        mu_te = Xte.mean(axis=1, keepdims=True); sig_te = Xte.std(axis=1, keepdims=True)
        sig_tr = np.where(sig_tr < 1e-8, 1.0, sig_tr)
        sig_te = np.where(sig_te < 1e-8, 1.0, sig_te)
        state.features["clean_train"] = (Xtr - mu_tr) / sig_tr
        state.features["clean_test"] = (Xte - mu_te) / sig_te
        state.features["norm_mu_train"] = mu_tr; state.features["norm_sigma_train"] = sig_tr
        state.features["norm_mu_test"] = mu_te; state.features["norm_sigma_test"] = sig_te
        state.target_base_train = (state.F_train - mu_tr) / sig_tr
        state.target_base_test = (state.F_test - mu_te) / sig_te
        state.current_target_train = state.target_base_train - state.cumulative_train()
        state.current_target_test = state.target_base_test - state.cumulative_test()
        state.log(self.name)
        return state


class DetrendToken:
    name = "detrend"
    kind = "cleaning"
    def _detrend(self, X):
        n, d = X.shape
        t = np.arange(d, dtype=np.float32)
        tc = t - t.mean()
        denom = np.sum(tc ** 2) + 1e-8
        out = np.zeros_like(X)
        slopes = np.zeros((n, 1), dtype=np.float32)
        intercepts = np.zeros((n, 1), dtype=np.float32)
        for i in range(n):
            xm = X[i].mean()
            slope = np.sum(tc * (X[i] - xm)) / denom
            intercept = xm - slope * t.mean()
            out[i] = X[i] - (slope * t + intercept)
            slopes[i, 0] = slope; intercepts[i, 0] = intercept
        return out, slopes, intercepts
    def apply(self, state):
        Xtr, Xte = latest_clean_pair(state)
        ctr, str_, itr = self._detrend(Xtr)
        cte, ste, ite = self._detrend(Xte)
        state.features["clean_train"] = ctr; state.features["clean_test"] = cte
        state.features["trend_slope_train"] = str_; state.features["trend_slope_test"] = ste
        state.log(self.name)
        return state


class FeatRawToken:
    name = "feat_raw"
    kind = "feature"
    def apply(self, state):
        Xtr, Xte = latest_clean_pair(state)
        state.features["model_input_train"] = Xtr.copy()
        state.features["model_input_test"] = Xte.copy()
        state.log(self.name)
        return state


class FeatLagToken:
    name = "feat_lag"
    kind = "feature"
    def _make(self, X, horizon):
        L = min(3 * horizon, X.shape[1], 512)
        recent = X[:, -L:]
        stats = np.column_stack([X.mean(axis=1), X.std(axis=1), X.min(axis=1), X.max(axis=1)])
        return np.hstack([recent, stats]).astype(np.float32)
    def apply(self, state):
        Xtr, Xte = latest_clean_pair(state)
        state.features["model_input_train"] = self._make(Xtr, state.F_train.shape[1])
        state.features["model_input_test"] = self._make(Xte, state.F_test.shape[1])
        state.log(self.name)
        return state


In [ ]:
def squared_distance_matrix(a, b):
    a2 = np.sum(a ** 2, axis=1, keepdims=True)
    b2 = np.sum(b ** 2, axis=1, keepdims=True).T
    return np.maximum(a2 + b2 - 2.0 * (a @ b.T), 0.0)

def median_lengthscale(X):
    D = np.sqrt(squared_distance_matrix(X, X))
    tri = D[np.triu_indices(X.shape[0], k=1)]
    tri = tri[tri > 0]
    return float(np.median(tri)) if tri.size else 1.0


class KernelRBFToken:
    name = "kernel_rbf"
    kind = "model"
    def __init__(self, ridge=1e-2):
        self.ridge = ridge
    def apply(self, state):
        Xtr = state.features["model_input_train"].astype(np.float32)
        Xte = state.features["model_input_test"].astype(np.float32)
        Y = state.current_target_train.astype(np.float32)
        ls = median_lengthscale(Xtr)
        K = np.exp(-squared_distance_matrix(Xtr, Xtr) / (2.0 * ls ** 2))
        Kte = np.exp(-squared_distance_matrix(Xte, Xtr) / (2.0 * ls ** 2))
        A = K + self.ridge * np.eye(K.shape[0], dtype=np.float32)
        alpha = np.linalg.solve(A, Y)
        pred_train = K @ alpha
        pred_test = Kte @ alpha
        state.features["kernel_lengthscale"] = np.array([ls], dtype=np.float32)
        state.push_prediction(pred_train, pred_test, self.name)
        state.log(self.name)
        return state


class RidgeToken:
    name = "ridge"
    kind = "model"
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    def apply(self, state):
        from sklearn.linear_model import Ridge
        Xtr = state.features["model_input_train"].astype(np.float32)
        Xte = state.features["model_input_test"].astype(np.float32)
        Y = state.current_target_train.astype(np.float32)
        model = Ridge(alpha=self.alpha)
        model.fit(Xtr, Y)
        pred_train = model.predict(Xtr)
        pred_test = model.predict(Xte)
        state.push_prediction(pred_train, pred_test, self.name)
        state.log(self.name)
        return state


class StopToken:
    name = "STOP"
    kind = "control"
    def apply(self, state):
        forecast = state.cumulative_test()
        if "norm_mu_test" in state.features:
            forecast = forecast * state.features["norm_sigma_test"] + state.features["norm_mu_test"]
        lo = state.H_test.min(axis=1, keepdims=True)
        hi = state.H_test.max(axis=1, keepdims=True)
        span = np.where(hi > lo, hi - lo, 1.0)
        forecast = np.clip(forecast, lo - 2 * span, hi + 2 * span)
        state.final_forecast = forecast
        state.mase = mase(state.F_test, forecast, state.H_test)
        state.features["final_forecast"] = forecast
        state.log(self.name)
        return state


TOKENS = {
    "normalize": NormalizeToken(),
    "detrend": DetrendToken(),
    "feat_raw": FeatRawToken(),
    "feat_lag": FeatLagToken(),
    "kernel_rbf": KernelRBFToken(ridge=1e-2),
    "ridge": RidgeToken(alpha=1.0),
    "STOP": StopToken(),
}

print("tokens:", list(TOKENS))


## 4. One Editable Chain Playground

Change CHAIN and rerun this cell. The chain should usually start with normalize, then one cleaning token if desired, then a feature token, then one or more model tokens, then STOP.


In [ ]:
from IPython.display import display, Markdown


def array_summary(x):
    x = np.asarray(x)
    return {
        "shape": tuple(x.shape),
        "mean": float(np.nanmean(x)),
        "std": float(np.nanstd(x)),
        "min": float(np.nanmin(x)),
        "max": float(np.nanmax(x)),
    }


def important_feature_summary(state):
    keys = [
        "clean_train", "clean_test",
        "model_input_train", "model_input_test",
        "last_prediction_train", "last_prediction_test",
        "current_residual_train", "current_residual_test",
        "final_forecast",
    ]
    rows = []
    for key in keys:
        if key in state.features:
            rows.append({"feature": key, **array_summary(state.features[key])})
    return pd.DataFrame(rows)


def step_trace_df(state):
    rows = []
    for i, row in enumerate(state.steps, start=1):
        rows.append({
            "step": i,
            "token": row["token"],
            "n_predictions": row["n_predictions"],
            "train_residual_norm": row["train_residual_norm"],
            "test_residual_norm": row["test_residual_norm"],
            "mase": row["mase"],
        })
    return pd.DataFrame(rows)


def plot_step_diagnostics(state, max_samples=2):
    trace = step_trace_df(state)
    if not trace.empty:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(trace["step"], trace["train_residual_norm"], marker="o", label="train residual")
        ax.plot(trace["step"], trace["test_residual_norm"], marker="s", label="test residual")
        ax.set_xticks(trace["step"])
        ax.set_xticklabels(trace["token"], rotation=25, ha="right")
        ax.set_ylabel("L2 norm")
        ax.grid(alpha=0.25)
        ax.legend()
        fig.tight_layout()
        plt.show()

    if state.final_forecast is not None:
        n_show = min(max_samples, state.H_test.shape[0])
        fig, axes = plt.subplots(n_show, 1, figsize=(10, 2.5 * n_show), squeeze=False)
        for i, ax in enumerate(axes[:, 0]):
            h_tail = state.H_test[i, -min(250, state.H_test.shape[1]):]
            ax.plot(np.arange(-len(h_tail), 0), h_tail, color="0.65", label="history tail")
            ax.plot(np.arange(state.F_test.shape[1]), state.F_test[i], color="black", label="actual")
            ax.plot(np.arange(state.final_forecast.shape[1]), state.final_forecast[i], color="tab:orange", ls="--", label="forecast")
            ax.axvline(0, color="tab:blue", ls=":", lw=1)
            ax.grid(alpha=0.2)
            ax.set_title(f"test sample {i}")
            if i == 0:
                ax.legend(fontsize=8)
        fig.tight_layout()
        plt.show()


def inspect_chain(chain, show_each_step=True):
    state = PlaygroundState(H_TRAIN, F_TRAIN, H_TEST, F_TEST)

    display(Markdown("### Chain"))
    display(pd.DataFrame({"position": range(1, len(chain) + 1), "token": chain}))

    for name in chain:
        if name not in TOKENS:
            raise KeyError(f"Unknown token {name!r}. Available: {list(TOKENS)}")

        TOKENS[name].apply(state)

        if show_each_step:
            display(Markdown(f"### After '{name}'"))
            display(step_trace_df(state).tail(1))
            display(Markdown("**Important arrays**"))
            display(important_feature_summary(state))
            plot_step_diagnostics(state)

    display(Markdown("### Final Trace"))
    display(step_trace_df(state))
    print("FINAL MASE:", state.mase)
    return state


def run_chain(chain, verbose=False):
    # Quiet version used by exhaustive search.
    state = PlaygroundState(H_TRAIN, F_TRAIN, H_TEST, F_TEST)
    for name in chain:
        if name not in TOKENS:
            raise KeyError(f"Unknown token {name!r}. Available: {list(TOKENS)}")
        TOKENS[name].apply(state)
    return state


CHAIN = [
    "normalize",
    "feat_raw",
    "kernel_rbf",
    "STOP",
]

state = inspect_chain(CHAIN, show_each_step=True)


In [ ]:
def plot_forecast(state, n_show=4):
    n_show = min(n_show, state.H_test.shape[0])
    fig, axes = plt.subplots(n_show, 1, figsize=(12, 2.5 * n_show), squeeze=False)
    for i, ax in enumerate(axes[:, 0]):
        h_tail = state.H_test[i, -min(300, state.H_test.shape[1]):]
        ax.plot(np.arange(-len(h_tail), 0), h_tail, color="0.6", label="history tail")
        ax.plot(np.arange(state.F_test.shape[1]), state.F_test[i], color="black", label="actual")
        ax.plot(np.arange(state.final_forecast.shape[1]), state.final_forecast[i], color="tab:orange", ls="--", label="forecast")
        ax.axvline(0, color="tab:blue", ls=":", lw=1)
        ax.grid(alpha=0.2)
        ax.set_title(f"test sample {i}")
        if i == 0:
            ax.legend()
    fig.tight_layout()

plot_forecast(state)


## 5. Exhaustive Search

This enumerates a small grammar by hand. It always starts with normalization, optionally adds detrending, chooses a feature token, then tries all model sequences up to MAX_MODEL_DEPTH. The table displays every result.


In [ ]:
CLEANING_PREFIXES = [
    ["normalize"],
    ["normalize", "detrend"],
]
FEATURES = ["feat_raw", "feat_lag"]
MODELS = ["kernel_rbf", "ridge"]
MAX_MODEL_DEPTH = 2

chains = []
for cleaning in CLEANING_PREFIXES:
    for feat in FEATURES:
        for depth in range(1, MAX_MODEL_DEPTH + 1):
            for model_seq in itertools.product(MODELS, repeat=depth):
                chains.append([*cleaning, feat, *model_seq, "STOP"])

print("candidate chains:", len(chains))


In [ ]:
rows = []
for chain in tqdm(chains, desc="exhaustive chains"):
    chain_str = " -> ".join(chain)
    try:
        s = run_chain(chain, verbose=False)
        rows.append({
            "chain": chain_str,
            "mase": s.mase,
            "n_models": sum(TOKENS[t].kind == "model" for t in chain),
            "n_tokens": len(chain),
            "train_residual_norm": float(np.linalg.norm(s.current_target_train)),
            "test_residual_norm": float(np.linalg.norm(s.current_target_test)),
            "error": None,
        })
    except Exception as exc:
        rows.append({
            "chain": chain_str,
            "mase": np.inf,
            "n_models": sum(t in MODELS for t in chain),
            "n_tokens": len(chain),
            "train_residual_norm": np.nan,
            "test_residual_norm": np.nan,
            "error": repr(exc),
        })

results_df = pd.DataFrame(rows).sort_values("mase", ascending=True).reset_index(drop=True)
results_df


In [ ]:
print("Best chain")
best_chain = results_df.loc[0, "chain"].split(" -> ")
print(results_df.loc[0, "chain"])
print("MASE:", results_df.loc[0, "mase"])

best_state = run_chain(best_chain, verbose=True)
plot_forecast(best_state)


## Notes

- This notebook uses a train/test split across examples to keep the final table less leaky than row-wise LOO over augmented siblings.
- The models still use fixed hyperparameters; once the playground feels right, the next step is a validation split or nested selection for model/token choices.
- To add a token, create a class with name and kind, implement apply(state), then add it to TOKENS.
